<a href="https://colab.research.google.com/github/vanha2301/DeepLearning-CV-base-VNAI/blob/main/Dataset_CIFAR10_Train_Test_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

vanha2301_test_dataset_cifar10_path = kagglehub.dataset_download('vanha2301/test-dataset-cifar10')

print('Data source import complete.')


## Import Lib ALL

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
from torchvision.transforms import Compose, ToTensor
import torch.optim as optim

## Downloading dataset CIFAR 10

In [ ]:
transform = Compose([
    ToTensor()
])

train_ds = CIFAR10(root="./data", train=True, download=True, transform=transform)
test_ds  = CIFAR10(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=128, shuffle=False, num_workers=2, pin_memory=True)


100%|██████████| 170M/170M [00:03<00:00, 48.6MB/s] 


## Model

In [ ]:
class ModelOfCIFA(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = self.make_block(in_channels=3,  out_channels=8)
        self.conv2 = self.make_block(in_channels=8,  out_channels=16)
        self.conv3 = self.make_block(in_channels=16, out_channels=32)

        self.fc1 = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(32 * 4 * 4, 256),
            nn.LeakyReLU()
        )
        self.fc2 = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def make_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(),

            nn.MaxPool2d(kernel_size=2)
        )

    def forward(self, x):
        x = self.conv1(x)      # (N,8,16,16)
        x = self.conv2(x)      # (N,16,8,8)
        x = self.conv3(x)      # (N,32,4,4)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = self.fc2(x)
        return x


In [ ]:
# =========================
# TRAIN / EVAL FUNCTIONS
# =========================
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = running_loss / total
    acc = correct / total
    return avg_loss, acc

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = running_loss / total
    acc = correct / total
    return avg_loss, acc


In [ ]:
# =========================
# MAIN TRAIN
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ModelOfCIFA(num_classes=10).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epochs = 5
best_acc = 0.0

for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    print(f"Epoch {epoch}/{epochs} | "
          f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
          f"test_loss={test_loss:.4f} test_acc={test_acc:.4f}")

    # save best
    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), "best_cifar_cnn.pth")

print("Best test acc:", best_acc)

Epoch 1/5 | train_loss=1.5804 train_acc=0.4206 | test_loss=1.2901 test_acc=0.5290
Epoch 2/5 | train_loss=1.2620 train_acc=0.5444 | test_loss=1.1272 test_acc=0.5955
Epoch 3/5 | train_loss=1.1487 train_acc=0.5919 | test_loss=1.0211 test_acc=0.6354
Epoch 4/5 | train_loss=1.0599 train_acc=0.6226 | test_loss=0.9286 test_acc=0.6786
Epoch 5/5 | train_loss=1.0122 train_acc=0.6446 | test_loss=0.9488 test_acc=0.6732
Best test acc: 0.6786


In [ ]:
from PIL import Image
from torchvision.transforms import Compose, ToTensor, Resize

classes = ["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"]

img = Image.open("/kaggle/input/test-dataset-cifar10/images.png").convert("RGB")

# Vì model bạn train trên CIFAR10 32x32 -> resize về 32x32
transform_img = Compose([
    Resize((32, 32)),
    ToTensor()
])

x = transform_img(img).unsqueeze(0).to(device)  # (1,3,32,32)

with torch.no_grad():
    logits = model(x)
    probs = torch.softmax(logits, dim=1)[0]
    pred = probs.argmax().item()

print("Pred class:", classes[pred])
print("Confidence:", float(probs[pred]))


Pred class: airplane
Confidence: 0.6348775625228882
